# 第七课｜第一个 RTL 神经元

上一课第一次读了 **寄存器传输级（Register-Transfer Level, RTL）**。今天只解决：
> **一次 neuron update 怎样拆成 combinational path 与 sequential register update？**

为避免偷偷决定尚未冻结的 LIF/fixed-point 细节，本课使用教学用 integrate-and-fire neuron，复用第四课的 accumulator + threshold + reset contract。它不是正式 `MOD-003`。


## 1. 概念账本

**已经知道：** register、clock edge、RTL module/port，并见过一个简单 `always_ff`。

**今天学习：** combinational path、`always_comb`，以及它怎样与 sequential `always_ff` 配合。

**只预告：** testbench / waveform 留到下一课。


## 2. 本课 contract

每个 cycle：读取旧 `membrane_v` → `candidate = membrane_v + input_current` → `candidate >= threshold` 产生 spike condition → spike 时 next state 取 reset，否则取 candidate → clock edge 保存。

本课固定使用 8-bit signed demo，测试向量刻意避开 overflow。正式 rounding / saturation / leak 仍由 RMD-002 / RMD-003 决定。


## 3. 先用 Python oracle 重放规则

先预测 threshold=4、输入 `[1,1,1,1,2,2]` 的 spike cycle，再运行。


In [ ]:
def tutorial_if_step(state, input_current, threshold, reset_value=0):
    candidate = state + input_current
    spike = candidate >= threshold
    next_state = reset_value if spike else candidate
    return next_state, spike, candidate

state = 0
for cycle, current in enumerate([1, 1, 1, 1, 2, 2]):
    next_state, spike, candidate = tutorial_if_step(state, current, 4)
    print(f'cycle={cycle}: state={state}, input={current}, candidate={candidate}, spike={spike}, next={next_state}')
    state = next_state


## 4. 结构图

```mermaid
flowchart LR
 REG["membrane_v register"] --> ADD["adder"]
 IN["input_current"] --> ADD
 ADD --> C["candidate"]
 C --> CMP[">= threshold"]
 TH["threshold"] --> CMP
 CMP --> SEL["choose reset or candidate"]
 C --> SEL
 R["reset_value"] --> SEL
 SEL --> NEXT["next_v"]
 NEXT --> REG
 CLK["clock edge"] -.-> REG
```


## 5. `always_comb` 与 `always_ff`

`always_comb` 描述不保存历史的 next-state 组合计算；`always_ff @(posedge clk)` 描述 clock edge 时把结果写进 register。

本课使用固定 8-bit 宽度，故意不在第一次 RTL neuron 里加入 parameter、显式 sign-extension 或正式 overflow policy。


## 6. 完整教学 RTL

```systemverilog
module tutorial_if_neuron (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_current,
    input  logic signed [7:0] threshold,
    input  logic signed [7:0] reset_value,
    output logic signed [7:0] membrane_v,
    output logic              spike
);
    logic signed [7:0] candidate;
    logic signed [7:0] next_v;
    logic spike_next;

    always_comb begin
        candidate = membrane_v + input_current;
        spike_next = candidate >= threshold;

        if (spike_next)
            next_v = reset_value;
        else
            next_v = candidate;
    end

    always_ff @(posedge clk) begin
        if (!rst_n) begin
            membrane_v <= reset_value;
            spike <= 1'b0;
        end else begin
            membrane_v <= next_v;
            spike <= spike_next;
        end
    end
endmodule
```


## 7. 逐段读

`candidate`、`spike_next`、`next_v` 都是当前 state/input 的组合结果。`membrane_v` 与 `spike` 在 clock edge 后更新。

`if (spike_next)` 对应一个硬件选择关系：reset 或 candidate。它不是 CPU 运行时才临时决定“要不要造一个 mux”。


### 可选实验：8-bit overflow 为什么会让直觉失效？

当前教学 RTL 的 `candidate` 只有 8-bit signed，范围是 `-128..127`。如果数学上得到的结果超出范围，硬件会保留有限位宽的 bit pattern；按 two's-complement 解释时就可能变成负数。

例如先预测 `100 + 50` 在 8-bit signed 中会变成什么，然后运行下面的 Python 小实验。

In [ ]:
def wrap_signed_8(x: int) -> int:
    return ((x + 128) % 256) - 128

mathematical_sum = 100 + 50
wrapped_sum = wrap_signed_8(mathematical_sum)
print('mathematical sum =', mathematical_sum)
print('8-bit signed result =', wrapped_sum)


这里会得到 `150 -> -106`。如果 threshold 是正数，当前教学 RTL 可能因此“不 spike”，这正是有限位宽 arithmetic 的危险之处。

**但不要把这个结果写进正式 golden test。** 本课明确把 overflow 排除在 contract 之外；wrap-around 只是当前 8-bit 教学实现的现象，不是我们已经批准的神经元语义。正式实现必须回到 RMD-002 / RMD-003，明确选择 widening、saturation、wrap-around 或其他 policy，再让 test oracle 固化该选择。

## 8. Try It：手算一个 edge

edge 前：`membrane_v=3, input_current=1, threshold=4, reset_value=0`。先写 candidate / spike_next / next_v，再写 edge 后 membrane_v / spike。


## 9. AI Task

让 AI 对照本课五条 contract，逐条指出 RTL 中对应的语句。如果发现不一致，只报告差异，不允许它改 contract。


## 10. Human Check

不用 AI，你应该能指出 combinational path 与 stored state；解释 `always_comb` / `always_ff` 的分工；说明为什么 `candidate=4` 与 edge 后 `membrane_v=0` 可以同时正确；说明为什么本教学模块不能冒充正式 LIF RTL。


## 11. Engineering Handoff

`rtl/learning/tutorial_if_neuron.sv` 只验证“会把已知 contract 映射到 RTL 结构”。正式 `rtl/neuron/lif_neuron_engine.sv` 仍等待 v0 semantics 与 fixed-point contract。


## 12. 项目追踪 Project Trace

- Lesson: `LSN-007`
- Mapping: `RMD-004` teaching precursor
- Formal `MOD-003`: intentionally not created


## 13. Exit Ticket

你能从 contract 画出 combinational path + registers，并从 RTL 指出两部分。下一课不再增加 neuron 规则，只学习怎样验证它。
